# Performing Principal Component Analysis for the Validation dataset

Importing relevant packages

In [ ]:
# Importing relevant packages
# Libraries for general file handling, plotting, and data structures
import os
import matplotlib.pyplot as plt
import geopandas # Library for handling geospatial vector data
import pandas as pd
import seaborn as sn # Library for statistical data visualization (used for heatmaps)
import numpy as np
from pyspatialml import Raster # Library for working with geospatial rasters

In [ ]:
# Libraries imported from scikit-learn for machine learning and data transformation
from sklearn.preprocessing import StandardScaler # For standardizing (scaling) data
from sklearn.pipeline import Pipeline # For chaining steps (scaling -> PCA -> Classification)
from sklearn.model_selection import cross_validate # For cross-validation (though not fully utilized here)
from sklearn.model_selection import train_test_split # For splitting data into training and testing sets
from sklearn.ensemble import RandomForestClassifier # The RF model used for testing PCA features
from sklearn.decomposition import PCA # The core library for Principal Component Analysis

Setting up the directory

In [ ]:
os.chdir(r'/workspace/MCC/') # Set the current working directory to the 'MCC' validation area data folder

Loading the data (landslide presence/absebce points)

In [ ]:
data=pd.read_csv('sample_points.csv') # Load the pre-processed MCC sample points (features and labels).

In [ ]:
data.head()

Separating features from labels

In [ ]:
X = data.iloc[:, 0:-1] # Select all columns except the last one as features (X).

In [ ]:
y = data["Landslide"] # Select the "Landslide" column as the target labels (y).

Splitting the data into training (70%) and testing (30%)

In [ ]:
# Split the MCC data into internal training/testing sets for PCA analysis and model evaluation.
# test_size=0.3: 30% of the MCC data is reserved for internal testing.
# stratify=y: Ensures a balanced split, maintaining the landslide/non-landslide ratio in both sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# It is also possible to manually split the dataset into training data and testing:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=[type here], random_state=42, stratify=y) # Final split using a fixed number of samples for the test set, ensuring class balance (stratify=y)

Standardizing the data before PCA

In [ ]:
scaler=StandardScaler().fit(X_train) # Initialize and fit the scaler ONLY on the training data to calculate mean/variance.
standardized_train_data = scaler.transform(X_train) # Scale the training data. Scaling gives a mean of 0 and standard deviation of 1.

Creating a PCA model. 16 Principal Components are chosen, because they retain 95% of variance in our data. It reduces the dimensionality of our dataset from 28 to 16 dimensions

In [ ]:
pca_model = PCA(n_components=12) # Initialize PCA to find 12 principal components (PCs).
pc_train_data= pca_model.fit_transform(standardized_train_data) # Fit PCA to the scaled training data and transform it into the 12-dimensional PC space.


In [ ]:
# Visualization of Explained Variance (Scree Plot)
# (Visual analysis to help determine the optimal number of PCs to keep)

plt.bar(range(1,13), pca_model.explained_variance_ratio_, # Bar plot: Variance explained by each individual PC.
        alpha=1,
        align='center',
       color='blue')
plt.step(range(1,13), np.cumsum(pca_model.explained_variance_ratio_), # Step plot: Cumulative variance explained as PCs are added.
         where='mid',
         color='red')
plt.ylabel('Explained variance ratio')
plt.xlabel('Principal Components')
plt.savefig(r'/workspace/fig/MCC/variance_pca.png', bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
pca_model.explained_variance_ratio_ # Prints the ratio of variance explained by each of the 12 components.

In [ ]:
pca_model.explained_variance_ratio_.cumsum() # Prints the total cumulative variance explained by the components.

The first component captures around 27% of variability in our data, while the second component - around 15% and so on.
16 Principal Components explain around 97% of the variability in the data

In [ ]:
# Component Loadings Analysis (How original features contribute to the PCs)

loadings = pca_model.components_ # 'components_' holds the eigenvectors, which are the loadings (weights) of the original features for each PC.
num_pc = pca_model.n_features_in_ # Original number of features.
pc_list = ["PC"+str(i) for i in list(range(1, num_pc+1))] 
loadings_df = pd.DataFrame.from_dict(dict(zip(pc_list, loadings))) # Create a DataFrame to organize loadings.
loadings_df['variable'] = X.columns # Add original feature names.
loadings_df = loadings_df.set_index('variable')
loadings_df # Display the table showing the contribution of each feature to each PC.

In [ ]:
# get correlation matrix plot for loadings
f, ax = plt.subplots(figsize=(30, 10))
ax = sn.heatmap(loadings_df, annot=True, cmap='vlag') # Visualizing the loadings table helps identify which features define each PC (e.g., strong positive/negative correlations).
plt.savefig(r'/workspace/fig/MCC/loadings_tal_pca.png', bbox_inches='tight', dpi=600)
plt.show()

A different way of plotting it

In [ ]:
f, ax = plt.subplots(figsize=(20, 10))
ax = sn.heatmap(pca_model.components_, # Another visualization of the component matrix.
                 cmap='vlag',
                 yticklabels=[ "PCA"+str(x) for x in range(1,pca_model.n_components_+1)],
                 xticklabels=list(X.columns),
                 cbar_kws={"orientation": "vertical"})
# plt.savefig(r'/workspace/fig/MCC/figures/loadings_pca.png', bbox_inches='tight')



Bioplot contains PCA loading plot which shows how much each variables effects a principal component.

PCA Loading Plot: All vectors start at origin and their projected values on components explains how much influence they have on that component. Angles between vectors indicate whether there is correlation between them - smaller angles tell about stronger correlation.

In [ ]:
def myplot(score, coeff, labels=None):
    # Function to create a custom Biplot for PC1 vs PC2.
    # It visualizes data points (scatter) and feature contributions (vectors).
    xs = score[:, 0] # PC1 scores
    ys = score[:, 1] # PC2 scores
    n = coeff.shape[0]

    # Define color mapping for classes (No-landslide=Yellow, Landslide=Red)
    color_map = {0: 'yellow', 1: 'red'}
    colors = [color_map[label] for label in y_train] 

    # Scatter plot of data points in the PC space
    scatter = plt.scatter(xs * scalex, ys * scaley, c=colors, s=100, zorder=0) 
    
    # Plotting feature vectors (loadings) from the origin
    for i in range(n):
        plt.arrow(0, 0, coeff[i, 0], coeff[i, 1], color='black', alpha=0.7, width=0.0015) 
        # Label the feature vectors
        plt.text(coeff[i, 0] * distance + x_offset, coeff[i, 1] * distance + y_offset, label, color='black', ha='center', va='center', fontsize=25, alpha=1, zorder=10)

 # Create legend for class labels (Landslide and No-landslide)
    handles = [plt.Line2D([], [], marker="o", ls="", color=color_map[yi], markersize=20) for yi in color_map]
    plt.legend(handles, ['No-landslide', 'Landslide'], loc='lower left', bbox_to_anchor=(0.70, 0.0), title_fontsize=28, fontsize=35)
    plt.xlabel("PC{}".format(1), fontsize=26)
    plt.ylabel("PC{}".format(2), fontsize=26)

    plt.xticks(fontsize=27)
    plt.yticks(fontsize=27)
    
    plt.grid()

f, ax = plt.subplots(figsize=(25, 25))
# Generate the biplot using the scores and loadings for the first two PCs.
myplot(pc_train_data[:, 0:2], np.transpose(pca_model.components_[0:2, :]), list(X.columns))
plt.savefig(r'/workspace/fig/MCC/loadings_plot_pca.png', bbox_inches='tight', dpi=300) # Save the biplot.


Transforming testing data

In [ ]:
standardized_test_data=scaler.transform(X_test)

In [ ]:
pc_test_data=pca_model.transform(standardized_test_data)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

The resulted Principal Components can be used for Machine Learning instead of the original data. The results below are though not represented in the project, as PCA was mainly used for understanding the data. 

## Random Forest

In [ ]:
# Number of trees in random forest 
n_estimators = [100, 200, 300, 500, 700, 1000, 1200]
# Number of features to consider at every split
max_features = ['auto', 'sqrt', 'log2']

In [ ]:
# Create the  grid
grid_rf = {'n_estimators': n_estimators,
               'max_features': max_features}
print(grid_rf)

In [ ]:
# Create a based model
rf = RandomForestClassifier()
# Instantiate the grid search model
grid_search_rf = GridSearchCV(estimator = rf, param_grid = grid_rf, cv = 10)


In [ ]:
grid_search_rf.fit(pc_train_data, y_train)

In [ ]:
grid_search_rf.best_params_

Creating the final RF classifier with the best hyperparameters

In [ ]:
scaler = StandardScaler()
pca = PCA(n_components = 11)
# set the tolerance to a large value to make the example faster
rf = RandomForestClassifier(random_state=42, max_features= 'sqrt', n_estimators=100)
pipeline_rf = Pipeline([('scaler', scaler), ('dims_reduction', pca), ('classifier', rf)])

In [ ]:
pipeline_rf.fit(X_train, y_train)

Creating an evaluation function

In [ ]:
from sklearn import metrics
#from sklearn.metrics import plot_confusion_matrix, accuracy_score, confusion_matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import cohen_kappa_score

def evaluate(model, test_features, test_labels):
    y_pred = model.predict(test_features)
    accuracy = metrics.accuracy_score(y_test, y_pred)
    print (accuracy)
    print(confusion_matrix(y_test,y_pred))
    print(cohen_kappa_score(y_test, y_pred))

In [ ]:
evaluate(pipeline_rf, X_test, y_test)

## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr = LogisticRegression() 

In [ ]:
param_grid_lr = [
  {'C': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000], 'solver': ['liblinear'], 'penalty': ['l1','l2']},
  {'C': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000], 'solver': ['lbfgs'], 'penalty': ['l2']},
 ]

In [ ]:
# Instantiate the grid search model
grid_search_LR = GridSearchCV(estimator = lr, param_grid = param_grid_lr, 
                          cv = 10, n_jobs = -1, verbose = 2)

# Fit the grid search to the data
grid_search_LR.fit(pc_train_data ,y_train)

In [ ]:
grid_search_LR.best_params_

Creating the final LR classifier with the best hyperparameters

In [ ]:
scaler = StandardScaler()
pca = PCA(n_components = 11)
# set the tolerance to a large value to make the example faster
lr = LogisticRegression(random_state=42, C=0.1, penalty='l2', solver='liblinear')
pipeline_lr = Pipeline([('scaler', scaler), ('dims_reduction', pca), ('classifier', lr)])

In [ ]:
pipeline_lr.fit(X_train, y_train)

In [ ]:
evaluate(pipeline_lr,X_test,y_test)

## Support Vector Machine

In [ ]:
from sklearn.svm import SVC

Parameters for  grid search

In [ ]:
param_grid_SVM = [
  {'C': [0.001, 0.1, 1, 10], 'kernel': ['linear']},
  {'C': [0.001, 0.1, 1, 10], 'gamma': [1, 0.1, 0.01,0.001, 0.0001], 'kernel': ['rbf', 'poly', 'sigmoid']},
 ]


In [ ]:
# Create a based model
svm = SVC(random_state=42)
# Instantiate the grid search model
grid_search_SVM = GridSearchCV(estimator = svm, param_grid = param_grid_SVM, 
                          cv = 10, n_jobs = -1, verbose = 2)

# Fit the grid search to the data
grid_search_SVM.fit(pc_train_data,y_train)

In [ ]:
grid_search_SVM.best_params_

Creating the final SVM classifier with the best hyperparameters

In [ ]:
scaler = StandardScaler()
pca = PCA(n_components = 5)
svm = SVC(random_state=42, C=10, gamma=0.01, kernel='sigmoid', probability=True)
pipeline_svm = Pipeline([('scaler', scaler), ('dims_reduction', pca), ('classifier', svm)])

In [ ]:
pipeline_svm.fit(X_train, y_train)

In [ ]:
evaluate(pipeline_svm, X_test ,y_test)

End.